# EP5 · Consultas en lenguaje natural con IA — DataRetail 360

**IDY1102 Estructuras y Visualización de Datos · Sección 003D · Unidad 3 · EP5**
Equipo: Anyara Rosso · Robert Groom

En este notebook probamos una herramienta de IA generativa para hacer consultas en lenguaje natural
sobre nuestro modelo. Le hacemos seis preguntas de negocio, la herramienta las traduce a SQL,
ejecutamos ese SQL sobre la capa Gold y comparamos el resultado con nuestra cifra de control.

La IA no ve los datos. Solo recibe la capa semántica (`docs/capa_semantica.json`), que describe las
vistas, las métricas y las reglas del modelo. Por eso el SQL se ejecuta aquí y no en el chat.

| Paso | Dónde | Cuántas veces |
|---|---|---|
| 0. Preparar la base | Este notebook | Una vez cada vez que se abre |
| 1. Dar contexto a la IA | Chat de la IA | Una vez por chat |
| 2. Preguntar, pegar el SQL, comprobar y anotar | Chat y notebook | Una vez por pregunta |

## 0. Preparación

Funciona en Google Colab y en Jupyter. En Colab clona el repositorio y genera la base Gold con el
pipeline, porque la base no está en GitHub (demora cerca de un minuto). Si la base ya existe, no la
vuelve a generar.

In [1]:
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/catluvies/dataretail360-ep3.git"

# Buscamos la carpeta del proyecto. Si no está (Colab), clonamos el repositorio.
aqui = Path.cwd()
RAIZ = next((p for p in [aqui, *aqui.parents] if (p / "src" / "pipeline.py").exists()), None)
if RAIZ is None:
    RAIZ = aqui / "dataretail360-ep3"
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "-q", REPO, str(RAIZ)], check=True)
os.chdir(RAIZ)

# La base Gold se genera con el pipeline del proyecto.
BASE = RAIZ / "data" / "gold" / "dwh_dataretail360.db"
if not BASE.exists():
    for modulo in (["src.generar_fuentes"], ["src.pipeline", "--todo", "--reiniciar"]):
        paso = subprocess.run([sys.executable, "-m", *modulo], capture_output=True, text=True)
        if paso.returncode != 0:
            print(paso.stdout[-2000:], paso.stderr[-2000:])
            raise RuntimeError(f"Falló {modulo[0]}")

print(f"Proyecto: {RAIZ.name}")
print(f"Base Gold lista: {BASE.stat().st_size / 1e6:.0f} MB")

Proyecto: dataretail360-ep3
Base Gold lista: 134 MB


Dos funciones para el resto del notebook:

- `consultar(sql)` ejecuta un SQL sobre la base Gold, en modo solo lectura, y devuelve una tabla.
- `comprobar(pregunta, resultado)` muestra esa tabla y compara su primera fila con la cifra de control.

La cifra de control de cada pregunta sale de nuestra consulta de referencia, escrita con las métricas
de la capa semántica. Los números se comparan redondeados a los mismos decimales de la cifra de control.

In [2]:
import html, numbers, sqlite3
import pandas as pd
from IPython.display import HTML, display

# Pregunta y consulta de referencia del equipo. P6 no tiene: el modelo no guarda campañas.
REFERENCIA = {
    "P1": ("¿Cuál fue la venta neta del primer semestre de 2026?",
           "SELECT SUM(venta_neta) FROM vw_ventas WHERE anio = 2026 AND semestre = 1"),
    "P2": ("¿Cuál fue el ticket promedio de junio de 2026?",
           """SELECT ROUND(SUM(venta_neta) * 1.0
                     / COUNT(DISTINCT CASE WHEN es_devolucion = 0 THEN numero_documento END))
              FROM vw_ventas WHERE anio_mes = '2026-06'"""),
    "P3": ("¿Qué sucursal tuvo la mayor tasa de quiebre de stock en el primer semestre de 2026?",
           """SELECT sucursal, ROUND(100.0 * SUM(es_quiebre) / COUNT(*), 2) AS tasa_quiebre
              FROM vw_inventario WHERE anio = 2026 AND semestre = 1
              GROUP BY sucursal ORDER BY tasa_quiebre DESC LIMIT 1"""),
    "P4": ("¿Para cuántos días alcanzaba el stock a fines de junio de 2026?",
           "SELECT dias_cobertura FROM vw_kpi_inventario_mensual WHERE anio_mes = '2026-06'"),
    "P5": ("¿Cuánto vendió el canal online en marzo de 2026?",
           "SELECT SUM(venta_neta) FROM vw_ventas WHERE tipo_canal = 'Digital' AND anio_mes = '2026-03'"),
    "P6": ("¿Qué campaña comercial generó más ventas en 2026?", None),
}
registro = {}   # lo que va quedando de cada pregunta respondida, para el resumen final


def consultar(sql):
    """Ejecuta un SQL sobre la base Gold (solo lectura) y devuelve una tabla."""
    sql = sql.replace("```sql", "").replace("```", "").strip()
    tabla, error = pd.DataFrame(), None
    if sql:
        conexion = sqlite3.connect(f"file:{BASE}?mode=ro", uri=True)
        try:
            cursor = conexion.execute(sql)
            columnas = [c[0] for c in cursor.description or []]
            tabla = pd.DataFrame(cursor.fetchall(), columns=columnas)
        except Exception as e:
            error = str(e)
        finally:
            conexion.close()
    tabla.attrs.update(sql=sql, error=error)
    return tabla


def _texto(valor):
    """Número o texto escrito como lo leemos: 2.112.212.190 · 10,93."""
    if valor is None or valor != valor:
        return "vacío"
    if isinstance(valor, numbers.Real):
        escrito = f"{valor:,.2f}".rstrip("0").rstrip(".") if valor % 1 else f"{valor:,.0f}"
        return escrito.translate(str.maketrans(",.", ".,"))
    return str(valor)


def _igual(valor, control):
    if isinstance(control, str):
        return str(valor).strip().lower() == control.lower()
    decimales = len(str(control).partition(".")[2].rstrip("0"))
    try:
        return round(float(valor), decimales) == round(float(control), decimales)
    except (TypeError, ValueError):
        return False


def comprobar(pregunta, resultado, observacion=""):
    """Muestra el resultado de la IA y lo compara con la cifra de control."""
    texto, sql_referencia = REFERENCIA[pregunta.split()[0]]
    sql, error = resultado.attrs.get("sql", ""), resultado.attrs.get("error")
    primera = resultado.iloc[0].tolist() if len(resultado) else []
    entrego = " · ".join(_texto(v) for v in primera)
    if len(resultado) > 1:
        entrego += f" (primera de {len(resultado)} filas)"

    if sql_referencia:
        control = consultar(sql_referencia).iloc[0].tolist()
        cifra = " · ".join(_texto(v) for v in control)
    else:
        control, cifra = None, "No se puede responder con estos datos"

    if not sql and control:
        estado, color, detalle = "Pendiente", "ambar", "Falta pegar el SQL de la IA."
    elif not sql:
        estado, color = "Sin SQL", "ambar"
        detalle = "Es lo esperado si la IA dijo que no se puede responder."
        if not observacion:
            detalle += " Anotar en la observación qué respondió."
        entrego = "No entregó SQL"
    elif error:
        estado, color, detalle, entrego = "Error al ejecutar", "rojo", error, f"Error: {error}"
        if "no such" in error:
            detalle += ". La IA usó una tabla o columna que no existe en el modelo (alucinación)."
    elif control is None:
        estado, color = "Revisar", "rojo"
        detalle = "El modelo no tiene campañas: este resultado no responde la pregunta."
    elif all(v is None for v in primera):
        estado, color, entrego = "No coincide", "rojo", "Sin datos"
        detalle = (f"La consulta no devolvió datos: revisar si el filtro está mal escrito. "
                   f"Cifra de control: {cifra}.")
    elif all(any(_igual(v, c) for v in primera) for c in control):
        estado, color, detalle = "Coincide", "verde", f"Cifra de control: {cifra}."
    else:
        estado, color = "No coincide", "rojo"
        detalle = f"Cifra de control: {cifra}. La IA entregó: {entrego}."

    if len(resultado):
        display(resultado.head(10))
    borde, fondo = {"verde": ("#1a7f37", "#dafbe1"), "rojo": ("#cf222e", "#ffebe9"),
                    "ambar": ("#9a6700", "#fff8c5")}[color]
    nota = (f"<br>Observación del equipo: {html.escape(observacion)}" if observacion
            else "<br>Falta la observación del equipo." if sql else "")
    display(HTML(f'<div style="padding:8px 12px;border-left:5px solid {borde};background:{fondo};'
                 f'color:#1f2328"><b>{pregunta} · {estado}.</b> {html.escape(detalle)}{nota}</div>'))

    if estado != "Pendiente":
        registro[pregunta] = {"Pregunta": texto, "SQL de la IA": sql,
                              "Resultado de la IA": entrego,
                              "Cifra de control": cifra, "Estado": estado,
                              "Observación": observacion}

## 1. Contexto para la IA

Primero anotamos qué herramienta usamos. Es lo único que se escribe en esta sección: va entre las comillas.

In [3]:
HERRAMIENTA = "Claude Opus 5.5 (Medium), en claude.ai"   # nombre y versión de la herramienta que usamos
FECHA = "2026-11-25"   # día en que hicimos las preguntas

En un chat nuevo adjuntamos `docs/capa_semantica.json` y pegamos el mensaje inicial. Se hace una
sola vez: mientras sigamos en el mismo chat, la herramienta lo recuerda.

La celda de abajo solo se ejecuta, no se le escribe nada. Imprime el mensaje para copiarlo al chat
y, en Colab, descarga el archivo que hay que adjuntar.

In [4]:
MENSAJE_INICIAL = """Vas a traducir preguntas de negocio a SQL para una base SQLite.
Usa solo las vistas y las reglas del archivo adjunto (capa semántica).
Entrega primero el SQL y después una explicación de una o dos líneas.
Si la pregunta no se puede responder con estos datos, dilo y no inventes tablas ni columnas."""
print(MENSAJE_INICIAL)

# En Colab descarga el archivo para adjuntarlo al chat; en Jupyter muestra dónde está.
try:
    from google.colab import files
    files.download("docs/capa_semantica.json")
except ImportError:
    print("\nArchivo para adjuntar: docs/capa_semantica.json")

Vas a traducir preguntas de negocio a SQL para una base SQLite.
Usa solo las vistas y las reglas del archivo adjunto (capa semántica).
Entrega primero el SQL y después una explicación de una o dos líneas.
Si la pregunta no se puede responder con estos datos, dilo y no inventes tablas ni columnas.

Archivo para adjuntar: docs/capa_semantica.json


## 2. Revisión antes de usar una respuesta

La regla 9 de la capa semántica pide que una persona revise cada respuesta antes de comunicarla.
En cada pregunta revisamos esto:

1. ¿El SQL usa solo las vistas permitidas?
2. ¿Usa la métrica oficial (`venta_neta`, `ticket_promedio`, `tasa_quiebre`, `dias_cobertura`)?
3. ¿Los filtros de fecha, canal y sucursal son los que pedía la pregunta?
4. ¿El resultado coincide con la cifra de control?
5. Si no devuelve filas, ¿es porque no hay datos o porque el filtro está mal escrito?

El punto 4 lo hace `comprobar`. Los demás los revisamos leyendo el SQL, y lo que encontramos queda
escrito en la observación de cada pregunta.

## 3. Las seis preguntas

Cada pregunta apunta a un error que la IA puede cometer con nuestro modelo. El SQL se pega tal como
lo entrega la herramienta, sin corregirlo: si viene con un error, ese error es parte de la evidencia.

En cada celda se escribe en dos lugares y lo demás no se toca: el SQL va entre las tres comillas
de `sql_p1 = """ """` y la observación entre las comillas de `observacion=""`.

### P1 · ¿Cuál fue la venta neta del primer semestre de 2026?

**Qué queremos ver:** que use la métrica `venta_neta` y no sume las ventas devueltas.

In [5]:
# SQL que entregó la IA, pegado entre las comillas sin corregirlo.
sql_p1 = """
SELECT SUM(venta_neta) AS venta_neta
FROM vw_ventas
WHERE anio = 2026
  AND semestre = 1;
"""
resultado_p1 = consultar(sql_p1)
comprobar("P1", resultado_p1, observacion=(
    "Coincide. Usó la métrica oficial venta_neta sobre vw_ventas y filtró el año y el semestre que pedía la pregunta."))

,venta_neta
0,2112212190


### P2 · ¿Cuál fue el ticket promedio de junio de 2026?

**Qué queremos ver:** que divida la venta por los documentos de venta y no promedie las líneas.

In [6]:
# SQL que entregó la IA, pegado entre las comillas sin corregirlo.
sql_p2 = """
SELECT SUM(venta_neta) * 1.0
       / NULLIF(COUNT(DISTINCT CASE WHEN es_devolucion = 0 THEN numero_documento END), 0)
       AS ticket_promedio
FROM vw_ventas
WHERE anio_mes = '2026-06';
"""
resultado_p2 = consultar(sql_p2)
comprobar("P2", resultado_p2, observacion=(
    "Coincide al redondear: entregó 257.756,41. Dividió la venta por los documentos sin devolución, que es la fórmula oficial, y no promedió líneas."))

,ticket_promedio
0,257756.411683


### P3 · ¿Qué sucursal tuvo la mayor tasa de quiebre de stock en el primer semestre de 2026?

**Qué queremos ver:** que calcule una tasa y no cuente los días en quiebre.

In [7]:
# SQL que entregó la IA, pegado entre las comillas sin corregirlo.
sql_p3 = """
SELECT sucursal,
       ROUND(100.0 * SUM(es_quiebre) / NULLIF(COUNT(*), 0), 2) AS tasa_quiebre
FROM vw_inventario
WHERE anio = 2026
  AND semestre = 1
GROUP BY sucursal
ORDER BY tasa_quiebre DESC
LIMIT 1;
"""
resultado_p3 = consultar(sql_p3)
comprobar("P3", resultado_p3, observacion=(
    "Coincide, pero no sirve como prueba: esta misma pregunta está entre los ejemplos de la capa semántica, con su SQL. La IA ya tenía la respuesta."))

,sucursal,tasa_quiebre
0,Iquique,10.93


### P4 · ¿Para cuántos días alcanzaba el stock a fines de junio de 2026?

**Qué queremos ver:** que tome el stock del último día del mes y no lo sume entre días.

In [8]:
# SQL que entregó la IA, pegado entre las comillas sin corregirlo.
sql_p4 = """
SELECT SUM(stock_cierre_mes) * 28.0
       / NULLIF(SUM(unidades_vendidas_28d), 0) AS dias_cobertura
FROM agg_inventario_mes
WHERE anio_mes = '2026-06';  -- columna supuesta
"""
resultado_p4 = consultar(sql_p4)
comprobar("P4", resultado_p4, observacion=(
    "Coincide al redondear (33,21). Usó la fórmula oficial, pero avisó que suponía la columna anio_mes porque la capa semántica no listaba las columnas de agg_inventario_mes. La columna existe; el vacío era de nuestra documentación."))

,dias_cobertura
0,33.206704


### P5 · ¿Cuánto vendió el canal online en marzo de 2026?

**Qué queremos ver:** que entienda "online" como los dos canales digitales (sitio web y app). En los datos no hay un canal que se llame "Online".

In [9]:
# SQL que entregó la IA, pegado entre las comillas sin corregirlo.
sql_p5 = """
SELECT SUM(venta_neta) AS venta_neta
FROM vw_ventas
WHERE canal IN ('Sitio web', 'App móvil')
  AND anio_mes = '2026-03';
"""
resultado_p5 = consultar(sql_p5)
comprobar("P5", resultado_p5, observacion=(
    "Coincide. Entendió «online» como Sitio web más App móvil, que son los nombres de canal de la regla 6, y no buscó un canal llamado Online."))

,venta_neta
0,92802720


### P6 · ¿Qué campaña comercial generó más ventas en 2026?

**Qué queremos ver:** que diga que no se puede responder. El modelo no tiene información de campañas, así que cualquier SQL o nombre de campaña sería inventado. Si la IA no entrega SQL, la celda queda vacía y en la observación va lo que respondió.

In [10]:
# SQL que entregó la IA, pegado entre las comillas sin corregirlo.
sql_p6 = """

"""
resultado_p6 = consultar(sql_p6)
comprobar("P6", resultado_p6, observacion=(
    "No entregó SQL. Respondió que la capa semántica no tiene tablas ni columnas de campañas y que no iba a suponer una. Es la respuesta correcta."))

## 4. Opcional · Las mismas preguntas sin la capa semántica

Para ver qué aporta la capa semántica, repetimos P3 y P4 en un chat nuevo, sin adjuntar el archivo.
A ese chat le damos solo el mensaje que imprime la celda de abajo, con el nombre de la vista y sus
columnas, sin métricas ni reglas.

In [11]:
columnas = consultar("SELECT name FROM pragma_table_info('vw_inventario')")["name"].tolist()
print("Vas a traducir preguntas de negocio a SQL para una base SQLite.")
print("La única vista disponible es vw_inventario, con estas columnas:")
print(", ".join(columnas))
print("Entrega primero el SQL y después una explicación de una o dos líneas.")

Vas a traducir preguntas de negocio a SQL para una base SQLite.
La única vista disponible es vw_inventario, con estas columnas:
sk_inventario, fecha, anio, semestre, trimestre, mes, nombre_mes, anio_mes, cod_sucursal, sucursal, tipo_sucursal, comuna_sucursal, region_sucursal, zona_sucursal, cod_producto, producto, marca, linea_producto, subcategoria, categoria, stock_cierre, unidades_recibidas, unidades_vendidas, stock_minimo, es_quiebre, id_carga
Entrega primero el SQL y después una explicación de una o dos líneas.


In [ ]:
sql_p3_sin_capa = """

"""
comprobar("P3 sin capa", consultar(sql_p3_sin_capa), observacion="")

In [ ]:
sql_p4_sin_capa = """

"""
comprobar("P4 sin capa", consultar(sql_p4_sin_capa), observacion="")

## 5. Resumen

La tabla junta lo que quedó en cada pregunta: el SQL de la IA, su resultado, la cifra de control,
si coincidió y nuestra observación.

In [14]:
resumen = pd.DataFrame(list(registro.values()), index=list(registro))
print(f"Herramienta: {HERRAMIENTA or 'pendiente'} · Fecha: {FECHA or 'pendiente'}")
with pd.option_context("display.max_colwidth", None):
    display(resumen)

Herramienta: Claude Opus 5.5 (Medium), en claude.ai · Fecha: 2026-11-25


,Pregunta,SQL de la IA,Resultado de la IA,Cifra de control,Estado,Observación
P1,¿Cuál fue la venta neta del primer semestre de 2026?,SELECT SUM(venta_neta) AS venta_neta\nFROM vw_ventas\nWHERE anio = 2026\n AND semestre = 1;,2.112.212.190,2.112.212.190,Coincide,Coincide. Usó la métrica oficial venta_neta sobre vw_ventas y filtró el año y el semestre que pedía la pregunta.
P2,¿Cuál fue el ticket promedio de junio de 2026?,"SELECT SUM(venta_neta) * 1.0\n / NULLIF(COUNT(DISTINCT CASE WHEN es_devolucion = 0 THEN numero_documento END), 0)\n AS ticket_promedio\nFROM vw_ventas\nWHERE anio_mes = '2026-06';","257.756,41",257.756,Coincide,"Coincide al redondear: entregó 257.756,41. Dividió la venta por los documentos sin devolución, que es la fórmula oficial, y no promedió líneas."
P3,¿Qué sucursal tuvo la mayor tasa de quiebre de stock en el primer semestre de 2026?,"SELECT sucursal,\n ROUND(100.0 * SUM(es_quiebre) / NULLIF(COUNT(*), 0), 2) AS tasa_quiebre\nFROM vw_inventario\nWHERE anio = 2026\n AND semestre = 1\nGROUP BY sucursal\nORDER BY tasa_quiebre DESC\nLIMIT 1;","Iquique · 10,93","Iquique · 10,93",Coincide,"Coincide, pero no sirve como prueba: esta misma pregunta está entre los ejemplos de la capa semántica, con su SQL. La IA ya tenía la respuesta."
P4,¿Para cuántos días alcanzaba el stock a fines de junio de 2026?,"SELECT SUM(stock_cierre_mes) * 28.0\n / NULLIF(SUM(unidades_vendidas_28d), 0) AS dias_cobertura\nFROM agg_inventario_mes\nWHERE anio_mes = '2026-06'; -- columna supuesta","33,21","33,2",Coincide,"Coincide al redondear (33,21). Usó la fórmula oficial, pero avisó que suponía la columna anio_mes porque la capa semántica no listaba las columnas de agg_inventario_mes. La columna existe; el vacío era de nuestra documentación."
P5,¿Cuánto vendió el canal online en marzo de 2026?,"SELECT SUM(venta_neta) AS venta_neta\nFROM vw_ventas\nWHERE canal IN ('Sitio web', 'App móvil')\n AND anio_mes = '2026-03';",92.802.720,92.802.720,Coincide,"Coincide. Entendió «online» como Sitio web más App móvil, que son los nombres de canal de la regla 6, y no buscó un canal llamado Online."
P6,¿Qué campaña comercial generó más ventas en 2026?,,No entregó SQL,No se puede responder con estos datos,Sin SQL,No entregó SQL. Respondió que la capa semántica no tiene tablas ni columnas de campañas y que no iba a suponer una. Es la respuesta correcta.


## 6. Conclusiones

**Qué resultó.** Con la capa semántica como contexto, Claude Opus 5.5 entregó un SQL que coincide con la cifra de control en las cinco preguntas que tenían respuesta. En la sexta dijo que faltaban datos de campañas y no inventó una tabla. En esta prueba no encontramos alucinaciones.

**Qué encontramos al revisar el SQL.** Que la cifra coincida no alcanza para dar por buena una respuesta. Al leer cada consulta con la lista de la sección 2 aparecieron dos cosas que la comparación de cifras no muestra:

1. **P3 no era una prueba real.** La pregunta está, con su SQL, entre los ejemplos de la capa semántica, así que la IA no tuvo que deducir nada. Fue un error nuestro al elegir las preguntas, y por eso P3 no cuenta como evidencia.
2. **A la capa semántica le faltaban las columnas de las vistas.** En P4 la IA avisó que suponía la columna `anio_mes`, y en P6 que no estaba documentado si la columna `descuento` existe en `vw_ventas`. Acertó, pero estaba adivinando. Lo corregimos: la versión 1.2 de `docs/capa_semantica.json` lista las columnas de las seis vistas. Esta prueba se hizo con la versión 1.1.

**Qué no probamos.** La sección 4, que repite dos preguntas sin la capa semántica, quedó sin hacer, y no repetimos las preguntas con la versión 1.2. Seis preguntas con buen contexto no permiten decir que la herramienta acierta siempre.

**Por qué revisamos antes de comunicar.** La IA escribe el SQL sin ver los datos y no sabe si el número que resulta es correcto. Eso solo se sabe ejecutando la consulta, comparando el resultado con una cifra que ya conocemos y leyendo el SQL para ver de dónde sale. Una respuesta que no pasa por esa revisión no se entrega al negocio.